<a href="https://colab.research.google.com/github/nishthadighe-bit/Data--Engineering-Practicals/blob/main/Practical_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests

def extract_api_data(url):
    """Fetches user data from a REST API and returns a DataFrame."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        # Normalizing nested JSON data into a flat DataFrame
        df = pd.json_normalize(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"API Error: {e}")
        return pd.DataFrame()

def extract_csv_data(file_path):
    """Reads a flat CSV file."""
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError as e:
        print(f"File Error: {e}")
        return pd.DataFrame()

# ---------------------------------------------------------
# Main Execution Pipeline
# ---------------------------------------------------------

# 1. API Extraction (Using free public JSONPlaceholder API)
api_url = "https://jsonplaceholder.typicode.com/users"
api_df = extract_api_data(api_url)

if not api_df.empty:
    # Select relevant columns and rename nested company column
    api_df = api_df[["id", "name", "email", "company.name"]]
    api_df.rename(columns={"company.name": "company"}, inplace=True)
    print("--- API Data Extracted ---")
    print(api_df.head(3))

# 2. Flat File Extraction (Creating & Reading local CSV)
csv_file = "locations.csv"

# Generating sample location data for matching IDs (1 to 10)
locations_data = {
    "id": list(range(1, 11)),
    "city": [
        "New York", "London", "Paris", "Tokyo", "Berlin",
        "Delhi", "Sydney", "Moscow", "Cairo", "Beijing"
    ],
    "country": [
        "USA", "UK", "France", "Japan", "Germany",
        "India", "Australia", "Russia", "Egypt", "China"
    ]
}

# Save dummy CSV to environment
pd.DataFrame(locations_data).to_csv(csv_file, index=False)

# Read CSV back
csv_df = extract_csv_data(csv_file)
print("\n--- Flat File CSV Extracted ---")
print(csv_df.head(3))

# 3. Data Transformation & Merging (ETL Pipeline Join)
if not api_df.empty and not csv_df.empty:
    merged_df = pd.merge(api_df, csv_df, on="id", how="inner")
    print("\n--- Merged ETL Pipeline Data View ---")
    print(merged_df.head())

    # Save to Target CSV File
    output_path = "cleaned_warehouse_profiles.csv"
    merged_df.to_csv(output_path, index=False)
    print(f"\nData successfully saved to '{output_path}'")

--- API Data Extracted ---
   id              name               email             company
0   1     Leanne Graham   Sincere@april.biz     Romaguera-Crona
1   2      Ervin Howell   Shanna@melissa.tv        Deckow-Crist
2   3  Clementine Bauch  Nathan@yesenia.net  Romaguera-Jacobson

--- Flat File CSV Extracted ---
   id      city country
0   1  New York     USA
1   2    London      UK
2   3     Paris  France

--- Merged ETL Pipeline Data View ---
   id              name                      email             company  \
0   1     Leanne Graham          Sincere@april.biz     Romaguera-Crona   
1   2      Ervin Howell          Shanna@melissa.tv        Deckow-Crist   
2   3  Clementine Bauch         Nathan@yesenia.net  Romaguera-Jacobson   
3   4  Patricia Lebsack  Julianne.OConner@kory.org       Robel-Corkery   
4   5  Chelsey Dietrich   Lucio_Hettinger@annie.ca         Keebler LLC   

       city  country  
0  New York      USA  
1    London       UK  
2     Paris   France  
3     Tokyo 